# Week 2 — Working with APIs

Covers: the `requests` library, environment variables (`.env`), and the `json` module.

This notebook uses a **mock LLM endpoint** (a local Python function standing in for a real
HTTP call) so it runs without network access or an API key. The `requests` syntax shown in
comments is exactly what you'd use against a real endpoint (OpenAI, Anthropic, or your
company's gateway) — only the URL and headers change.

## 1. Environment variables and `.env` files

In [ ]:
import os

# In a real project, secrets never go in code. They go in a .env file (which is
# git-ignored) and get loaded at runtime.
env_contents = 'LLM_API_KEY=demo-key-not-real\nLLM_MODEL=gpt-demo\n'

with open(".env", "w") as f:
    f.write(env_contents)

from dotenv import load_dotenv
load_dotenv()   # reads .env into the process environment

api_key = os.environ.get("LLM_API_KEY")
model = os.environ.get("LLM_MODEL")
print("Loaded model:", model)
print("Key starts with:", api_key[:4], "..." if api_key else "(missing)")

## 2. The shape of a real API call (reference — do not run against a real key here)

In [ ]:
# This is what a REAL call looks like. It is commented out deliberately —
# do not uncomment with a real key in a shared notebook.

# import requests
# response = requests.post(
#     "https://api.openai.com/v1/chat/completions",
#     headers={"Authorization": f"Bearer {api_key}"},
#     json={
#         "model": model,
#         "messages": [{"role": "user", "content": "Classify this support ticket."}],
#     },
#     timeout=30,
# )
# response.raise_for_status()          # raises an exception on 4xx/5xx
# data = response.json()               # parses the JSON body into a dict
print("See the commented block above for real `requests` usage.")

## 3. A runnable mock, same shape as the real call

In [ ]:
import json

def mock_llm_endpoint(payload: dict) -> dict:
    """Stands in for `requests.post(...).json()` — same input/output shape."""
    user_msg = payload["messages"][0]["content"]
    return {
        "id": "chatcmpl-demo",
        "model": payload["model"],
        "choices": [
            {"message": {"role": "assistant",
                          "content": f"[mock reply to]: {user_msg}"}}
        ],
        "usage": {"prompt_tokens": len(user_msg.split()), "completion_tokens": 6},
    }

request_payload = {
    "model": model,
    "messages": [{"role": "user", "content": "Classify this support ticket: refund request"}],
}

response_data = mock_llm_endpoint(request_payload)
print(json.dumps(response_data, indent=2))

## 4. Parsing the response — the part every LLM call needs

In [ ]:
reply_text = response_data["choices"][0]["message"]["content"]
tokens_used = response_data["usage"]["prompt_tokens"] + response_data["usage"]["completion_tokens"]

print("Reply:", reply_text)
print("Total tokens:", tokens_used)

# Defensive access — real responses occasionally omit fields on error paths.
# .get() with a default avoids a KeyError crash mid-pipeline.
safe_usage = response_data.get("usage", {}).get("prompt_tokens", 0)
print("Safe prompt token read:", safe_usage)

In [ ]:
import os
os.remove(".env")   # cleanup for this demo